In [20]:
import re
from typing import Dict, List
from transformers import pipeline, AutoTokenizer, AutoModelForTokenClassification

# Regex para PII estruturadas no padrão brasileiro
REGEX_PATTERNS = {
    "CPF": r"\b\d{3}\.\d{3}\.\d{3}-\d{2}\b|\b\d{11}b",
#    "EMAIL": r"\b[A-Za-Z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b",
    "EMAIL": r"\b[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+[a-zA-Z]{2,}\b",
    "TELEFONE": r"\b(?:\+55\s?)?(?:\(?\d{2}\)?\s?)?(?:9\d{4}|\d{4})[-\s]?\d{4}\b"
}

print("'Módulo de Regex para PII estruturadas carregado sem erros...")

print("Carregando o modelo de BERTimbau oficial NER...")

MODEL_NAME = "neuralmind/bert-base-portuguese-cased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME)

ner_pipeline = pipeline(
    "ner",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="first"
)

print("Modelo carregado com sucesso!.. ")

def unificar_subtokens(ner_results: List[Dict]) -> List[Dict]:
    """
    Agrupa sub-tokens do BERTimbau iniciados por '##' para reconstituir
    palavras e nomes completos identificados pelo modelo NER.
    """
    entidades_processadas = []
    entidade_atual = None

    for item in ner_results:
        palavra = item.get('word', '')
        entity_group = item.get('entity_group', item.get('entity', '0'))
        score = item.get('score', 0.0)
        

        #ignora rótulos de fora/neutros ('0')
        if entity_group == '0':
            continue
        
        # Se for um sub-token (inicia com ##), anexa à palavra anterior
        if palavra.startswith("##"):
            if entidade_atual:
                entidade_atual["valor"] += palavra.replace("##", "")
                entidade_atual["score_minimo"] = min(entidade_atual["score_minimo"], score)
        else:
            # Se já tínhamos uma entidade em construção, salvamos
            if entidade_atual:
                entidades_processadas.append(entidade_atual)
            
            # Inicia uma nova entidade
            entidade_atual = {
                "valor": palavra,
                "tipo": entity_group,
                "score_minimo": score
            }

    if entidade_atual:
        entidades_processadas.append(entidade_atual)

    return entidades_processadas


def anonimizar_texto(texto: str) -> Dict[str, object]:
    """
    Identifica PII estruturadas (Regex) e PII não estruturadas (BERTimbau NER)
    com recomposição de sub-tokens e aplica as tags de mascaramento LGPD.
    """
    texto_anonimizado = texto
    entidades_encontradas = []

    # 1. Detecção por Regex (CPFs, E-mails, Telefones)
    for tipo_pii, pattern in REGEX_PATTERNS.items():
        matches = list(re.finditer(pattern, texto_anonimizado))
        for match in matches:
            valor = match.group()
            entidades_encontradas.append({"tipo": tipo_pii, "valor": valor, "metodo": "Regex"})
            texto_anonimizado = texto_anonimizado.replace(valor, f"[MASCARADO: {tipo_pii}]")

    # 2. Detecção por NLP / NER com recomposição de sub-tokens
    ner_results = ner_pipeline(texto_anonimizado)
    entidades_recompostas = unificar_subtokens(ner_results)

    # Tipos de entidades validos para LGPD
    ENTIDADES_VALIDAS = {"PER", "ORG", "PESSOA", "LOCAL", "ORGANIZACAO"}

    for ent in entidades_recompostas:
        valor = ent["valor"]
        tipo = ent["tipo"]
        
        # Filtra ruídos ou marcadores de mascaramento já aplicados pelo Regex
        if tipo in ENTIDADES_VALIDAS and len(valor) > 2:
            if not any(tag in valor for tag in ["[MASCARADO:", "CPF", "EMAIL", "TELEFONE"]):
                entidades_encontradas.append({
                    "tipo": f"ENTIDADE_{tipo}",
                    "valor": valor,
                    "metodo": "BERTimbau NER"
                })
                texto_anonimizado = re.sub(rf"\b{re.escape(valor)}\b", f"[MASCARADO: {tipo}]", texto_anonimizado)

    return {
        "texto_original": texto,
        "texto_anonimizado": texto_anonimizado,
        "pii_detectadas": entidades_encontradas
    }

# Texto teste
texto_exemplo = ( 
    "O voluntario Juninho Silva, portador do CPF 123.456.789-00 e e-mail juningameplays.silva@email.com, "
    "Solicitou a autorização para envio de dados do projeto de apoio comunitario em Manaus."
)

resultado = anonimizar_texto(texto_exemplo)

print(">>>TEXTO ORIGINAL<<<")
print(resultado["texto_original"])
print("\n>>> TEXTO ANONIMIZADO <<<")
print(resultado["texto_anonimizado"])
print("\n>>> PII DETECTADAS <<<")
for pii in resultado["pii_detectadas"]:
    print(f"- [{pii['metodo']}] {pii['tipo']}: {pii['valor']}")

'Módulo de Regex para PII estruturadas carregado sem erros...
Carregando o modelo de BERTimbau oficial NER...


Loading weights: 100%|██████████████████████████████████████████████████| 197/197 [00:00<00:00, 33630.91it/s]
[transformers] BertForTokenClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias          

Modelo carregado com sucesso!.. 
>>>TEXTO ORIGINAL<<<
O voluntario Juninho Silva, portador do CPF 123.456.789-00 e e-mail juningameplays.silva@email.com, Solicitou a autorização para envio de dados do projeto ded apoio comunitario em Manaus.

>>> TEXTO ANONIMIZADO <<<
O voluntario Juninho Silva, portador do CPF [MASCARADO: CPF] e e-mail [MASCARADO: EMAIL], Solicitou a autorização para envio de dados do projeto ded apoio comunitario em Manaus.

>>> PII DETECTADAS <<<
- [Regex] CPF: 123.456.789-00
- [Regex] EMAIL: juningameplays.silva@email.com
